# Experiment 2: Statistical Significance & Hypothesis Testing

In this notebook, we perform the formal statistical analysis to validate the core hypotheses of the paper:
1. **Wilcoxon Signed-Rank Test**: Is there significant accuracy degradation or stagnation when oversampling (e.g., $K=2048 \to K=4096$) in the saturated phase?
2. **Spearman Correlation**: Does the saturation ratio $S(K)$ predict the marginal accuracy gain?
3. **Pearson Correlation (Decoupling)**: Are task difficulty (peak accuracy) and geometric complexity (erank asymptote) decoupled?

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.stats as stats
import sys

# Add src to python path to allow importing modules
sys.path.append(os.path.abspath('..'))

from src.datasets import load_all_datasets
from src.experiment import sample_and_evaluate, load_results
from src.visualization import plot_decoupling_hypothesis, plot_saturation_vs_marginal_gain

## Load Cached Sweeps Results

In [ ]:
results_path = '../results/all_results.json'
all_results = load_results(results_path)
if all_results is None:
    raise FileNotFoundError(f"Run the first notebook or 'run_all.py' script first to populate '{results_path}'")

## 1. Wilcoxon Signed-Rank Test (Oversampling Harm)

We test if there is a significant difference (or even a negative marginal return) when training size $K$ increases from $2048$ to $4096$ on the MNIST 3v8 task (which shows strong saturation).

In [ ]:
# Get MNIST data
datasets = load_all_datasets(data_dir='../data')
X_mnist, y_mnist = datasets['MNIST']
mask = (y_mnist == 3) | (y_mnist == 8)
X_38 = X_mnist[mask]
y_38 = y_mnist[mask]
y_38 = np.where(y_38 == 3, 0, 1)

# Standardize and PCA
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
scaler = StandardScaler()
X_38_s = scaler.fit_transform(X_38)
pca = PCA(n_components=50)
X_38_pca = pca.fit_transform(X_38_s)

# Evaluate K=2048 and K=4096 across 50 trials
accs_2048 = []
accs_4096 = []
for trial in range(50):
    acc_2k, _ = sample_and_evaluate(X_38_pca, y_38, K=2048, test_size=200, seed=trial)
    acc_4k, _ = sample_and_evaluate(X_38_pca, y_38, K=4096, test_size=200, seed=trial)
    accs_2048.append(acc_2k)
    accs_4096.append(acc_4k)

accs_2048 = np.array(accs_2048)
accs_4096 = np.array(accs_4096)

# One-tailed Wilcoxon test: Is K=2048 accuracy significantly greater than K=4096?
w_stat, p_val = stats.wilcoxon(accs_2048, accs_4096, alternative='greater')
print(f"Wilcoxon Signed-Rank Test (MNIST 3v8, K=2048 vs K=4096):")
print(f"  W-statistic = {w_stat}")
print(f"  p-value     = {p_val:.4f}")
print(f"  Mean Acc K=2048 = {accs_2048.mean():.4f}")
print(f"  Mean Acc K=4096 = {accs_4096.mean():.4f}")

## 2. Spearman Correlation: $S(K)$ vs. Marginal Accuracy Gain

We check if the saturation index $S(K) = \text{erank} / K$ is correlated with the marginal accuracy gain across all tasks and $K$ values.

In [ ]:
all_S = []
all_marginal = []

for task_name, results in all_results.items():
    for i, r in enumerate(results):
        if i == 0:  # Skip K=2 as there is no previous K to compute marginal gain
            continue
        all_S.append(r['S'])
        all_marginal.append(r['marginal'])

rho, p_value = stats.spearmanr(all_S, all_marginal)
print(f"Spearman Correlation (S vs Marginal Gain):")
print(f"  rho     = {rho:.3f}")
print(f"  p-value = {p_value:.2e}")
print(f"  N       = {len(all_S)} observations")

We can stratify these results by the predicted phase according to $S$:
- **Saturation**: $S < 0.3$
- **Transition**: $0.3 \le S < 1.0$
- **Exploration**: $S \ge 1.0$

In [ ]:
df = pd.DataFrame({'S': all_S, 'marginal': all_marginal})
df['phase'] = pd.cut(df['S'], bins=[0, 0.3, 1.0, np.inf], 
                     labels=['Saturation', 'Transition', 'Exploration'])
print("Mean marginal accuracy gain by predicted phase:")
print(df.groupby('phase', observed=False)['marginal'].agg(['mean', 'std', 'count']))

In [ ]:
%matplotlib inline
plot_saturation_vs_marginal_gain(all_results, '../figures/saturation_vs_marginal.png')

from IPython.display import Image
Image(filename='../figures/saturation_vs_marginal.png')

## 3. Pearson Correlation: Decoupling Hypothesis

We test if task difficulty (peak accuracy) and geometric complexity (erank asymptote $\text{erank}_\infty$) are correlated. A non-significant correlation supports the decoupling hypothesis.

In [ ]:
eranks_inf = []
peaks = []
task_names = []

for name, results in all_results.items():
    eranks_inf.append(results[-1]['mean_erank'])
    peaks.append(max([r['mean_acc'] for r in results]))
    task_names.append(name)

r_stat, p_val = stats.pearsonr(eranks_inf, peaks)
print(f"Pearson Correlation (erank_inf vs Peak Acc):")
print(f"  r-coefficient = {r_stat:.3f}")
print(f"  p-value       = {p_val:.3f}")
if p_val > 0.05:
    print("  Result: NOT SIGNIFICANT (supports the decoupling hypothesis)")
else:
    print("  Result: SIGNIFICANT (contradicts the decoupling hypothesis)")

In [ ]:
%matplotlib inline
plot_decoupling_hypothesis(all_results, '../figures/decoupling.png')

from IPython.display import Image
Image(filename='../figures/decoupling.png')